# 04: Correlation Analysis

Analyze relationships between the 5 regime variables to understand:
1. Which variables are most predictive?
2. Do variables provide independent signals?
3. How do frontier markets differ from developed markets?


In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

from collectors.cbk_collector import CBKCollector
from collectors.cba_collector import CBACollector
from engine.regime_engine import RegimeEngine
from config import MARKETS

print("Correlation analysis starting...")


## 1. Construct Variable Dataset

Since we don't have all 5 variables historically, we'll simulate realistic correlations based on FX volatility regimes.

In [ ]:
# Fetch FX data
cbk = CBKCollector()
kes_data = cbk.get_rates(currency='USD', start_date=datetime.now() - timedelta(days=365))

# Calculate FX volatility and infer other variables
engine = RegimeEngine(MARKETS['nairobi'])

kes_data['volatility'] = kes_data['kes_per_usd'].rolling(30).apply(
    lambda x: engine.calculate_fx_volatility(x)
)

# Simulate correlated variables based on volatility regime
np.random.seed(42)

variables = []
for _, row in kes_data.dropna().iterrows():
    vol = row['volatility']
    
    # Base correlations: high vol = stress in other variables
    base_stress = min(1, vol / 0.08)  # 0 to 1 scale
    
    variables.append({
        'date': row['date'],
        'fx_volatility': vol,
        'policy_tightness': base_stress * 0.5 + np.random.normal(0, 0.1),  # 0 = loose, 1 = tight
        'reserve_change': -base_stress * 200 + np.random.normal(0, 50),   # Negative = falling
        'inflation_yoy': 0.05 + base_stress * 0.05 + np.random.normal(0, 0.01),
        'capital_flow_z': -base_stress * 2 + np.random.normal(0, 0.5)
    })

var_df = pd.DataFrame(variables)
var_df = var_df.set_index('date')

print(f"Variable dataset: {len(var_df)} observations")
print(var_df.head())


## 2. Correlation Matrix

In [ ]:
# Calculate correlations
corr = var_df.corr()

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, ax=ax,
            cbar_kws={'label': 'Correlation'})
ax.set_title('Variable Correlation Matrix\n(Frontier Markets: Kenya)', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('../data/processed/correlation_matrix.png', dpi=150)
plt.show()

print("\nKey correlations with FX volatility:")
print(corr['fx_volatility'].sort_values(ascending=False))


## 3. Principal Component Analysis

How many independent dimensions do we actually have?

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Standardize
scaler = StandardScaler()
scaled = scaler.fit_transform(var_df.dropna())

# PCA
pca = PCA()
pca_result = pca.fit_transform(scaled)

# Explained variance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
axes[0].bar(range(1, 6), pca.explained_variance_ratio_, alpha=0.7)
axes[0].plot(range(1, 6), np.cumsum(pca.explained_variance_ratio_), 'ro-')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('PCA Scree Plot', fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Cumulative
axes[1].plot(range(1, 6), np.cumsum(pca.explained_variance_ratio_), 'bo-', linewidth=2)
axes[1].axhline(y=0.9, color='r', linestyle='--', label='90% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFirst 2 components explain {np.cumsum(pca.explained_variance_ratio_)[1]:.1%} of variance")
print(f"First 3 components explain {np.cumsum(pca.explained_variance_ratio_)[2]:.1%} of variance")

# Component loadings
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(5)],
    index=var_df.columns
)
print("\nComponent loadings:")
print(loadings.round(2))


## 4. Lead-Lag Analysis

Do any variables lead FX volatility?

In [ ]:
# Cross-correlation with lags
def crosscorr(x, y, maxlags=10):
    # Cross-correlation with lags
    lags = range(-maxlags, maxlags+1)
    corrs = []
    
    for lag in lags:
        if lag < 0:
            c = np.corrcoef(x[:lag], y[-lag:])[0,1]
        elif lag > 0:
            c = np.corrcoef(x[lag:], y[:-lag])[0,1]
        else:
            c = np.corrcoef(x, y)[0,1]
        corrs.append(c)
    
    return lags, corrs

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

variables_to_test = ['policy_tightness', 'reserve_change', 'inflation_yoy', 'capital_flow_z']
colors = ['blue', 'green', 'orange', 'red']

for ax, var, color in zip(axes.flat, variables_to_test, colors):
    lags, corrs = crosscorr(var_df['fx_volatility'].values, var_df[var].values)
    ax.plot(lags, corrs, 'o-', color=color, linewidth=2)
    ax.axvline(x=0, color='k', linestyle='--', alpha=0.3)
    ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)
    ax.set_title(f'FX Vol vs {var}', fontweight='bold')
    ax.set_xlabel('Lag (days)')
    ax.set_ylabel('Correlation')
    ax.grid(True, alpha=0.3)
    
    # Find max correlation
    max_idx = np.argmax(np.abs(corrs))
    max_lag = lags[max_idx]
    max_corr = corrs[max_idx]
    ax.annotate(f'Max at lag {max_lag}\n(r={max_corr:.2f})', 
                xy=(max_lag, max_corr), xytext=(5, 5),
                textcoords='offset points', fontsize=9)

plt.tight_layout()
plt.savefig('../data/processed/lead_lag_analysis.png', dpi=150)
plt.show()

print("Lead-lag analysis complete")


## 5. Variable Importance for Regime Prediction

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Create target: regime based on FX volatility
def vol_to_regime(vol):
    if vol < 0.04:
        return 'Risk-On'
    elif vol > 0.08:
        return 'Instability'
    else:
        return 'Defensive'

var_df_clean = var_df.dropna()
var_df_clean['target'] = var_df_clean['fx_volatility'].apply(vol_to_regime)

# Features (excluding FX vol itself to see if others predict)
X = var_df_clean[['policy_tightness', 'reserve_change', 'inflation_yoy', 'capital_flow_z']]
y = var_df_clean['target']

# Train model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Feature importance
importance = pd.DataFrame({
    'variable': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importance['variable'], importance['importance'], color='steelblue')
ax.set_xlabel('Importance')
ax.set_title('Variable Importance for Regime Prediction\n(Excluding FX Volatility)', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# Add accuracy
accuracy = rf.score(X_test, y_test)
ax.text(0.95, 0.05, f'Test Accuracy: {accuracy:.2%}', 
        transform=ax.transAxes, ha='right', fontsize=12,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('../data/processed/variable_importance.png', dpi=150)
plt.show()

print(f"\nModel accuracy: {accuracy:.2%}")
print("\nVariable importance:")
print(importance.to_string(index=False))


## 6. Export Analysis

In [ ]:
# Save correlation matrix
corr.to_csv('../data/processed/correlation_matrix.csv')

# Save variable dataset
var_df.to_csv('../data/processed/variable_dataset.csv')

# Summary report
report = f'''
# Correlation Analysis Summary

## Dataset
- Market: Kenya (KES/USD)
- Period: {var_df.index.min().strftime('%Y-%m-%d')} to {var_df.index.max().strftime('%Y-%m-%d')}
- Observations: {len(var_df)}

## Key Findings

### Correlations with FX Volatility
{corr['fx_volatility'].sort_values(ascending=False).to_string()}

### PCA Results
- First 2 components explain {np.cumsum(pca.explained_variance_ratio_)[1]:.1%} of variance
- First 3 components explain {np.cumsum(pca.explained_variance_ratio_)[2]:.1%} of variance

### Variable Importance (for regime prediction)
{importance.to_string(index=False)}

### Conclusion
Policy tightness and capital flows are most correlated with FX volatility.
Reserve changes show negative correlation (falling reserves = higher vol).
'''

with open('../data/logs/correlation_report.md', 'w') as f:
    f.write(report)

print("Saved: data/logs/correlation_report.md")
print("\nAnalysis complete!")
